# 03 - Feature Engineering

Builds the derived satellite flags, standard date/categorical features, assembles the final feature matrix, and performs the leakage-safe spatial-block train/test split.

In [1]:
# --- Notebook-chaining glue code (added when splitting the original pipeline) ---
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = pd.read_csv("preprocessing_output.csv")
print(f"Loaded {len(df)} rows from preprocessing_output.csv")

Loaded 92869 rows from preprocessing_output.csv


In [2]:
# Derived flags from the wildland-urban-interface / fire-type literature discussed:
# vegetation presence is a strong wildfire signal; low-vegetation + built-up is
# a strong industrial signal; an NBR drop indicates a likely burn scar.
df["is_vegetated"] = (df["ndvi"] > 0.3).astype(int)
df["is_built_up"] = (df["ndbi"] > 0.0).astype(int)
df["burn_signal"] = (df["nbr"] < -0.1).astype(int)
df[["ndvi", "ndbi", "nbr", "is_vegetated", "is_built_up", "burn_signal"]].describe()

,ndvi,ndbi,nbr,is_vegetated,is_built_up,burn_signal
count,92869.000000,92869.000000,92869.000000,92869.000000,92869.000000,92869.000000
mean,0.365490,-0.028681,0.162424,0.585319,0.480774,0.190849
std,0.248622,0.182115,0.265151,0.492670,0.499633,0.392973
min,-1.000000,-0.706683,-1.000000,0.000000,0.000000,0.000000
25%,0.106072,-0.178254,-0.048920,0.000000,0.000000,0.000000
50%,0.389542,-0.011440,0.164050,1.000000,0.000000,0.000000
75%,0.569077,0.119305,0.386107,1.000000,1.000000,0.000000
max,0.939827,1.000000,0.848965,1.000000,1.000000,1.000000


## Step 3: Standard feature engineering (dates, categorical encoding)

In [3]:
df["acq_date"] = pd.to_datetime(df["acq_date"])
df["month"] = df["acq_date"].dt.month
df["day_of_year"] = df["acq_date"].dt.dayofyear
df["hour"] = (df["acq_time"] // 100).astype(int)

for c in ["dist_to_industrial_km", "dist_to_farmland_km", "dist_to_mining_km"]:
    if c in df.columns:
        df[c] = df[c].fillna(df[c].max() if df[c].notna().any() else 999)

df["industrial_polygon_type"] = df["industrial_polygon_type"].fillna("none")
df["land_cover_class"] = df["land_cover_class"].fillna("unknown")

cat_cols = ["confidence", "daynight", "satellite", "instrument",
            "land_cover_class", "industrial_polygon_type"]
encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    df[c + "_enc"] = le.fit_transform(df[c].astype(str))
    encoders[c] = le
print("Categorical columns encoded.")

Categorical columns encoded.


In [4]:
# region/country deliberately excluded from FEATURES (geography-leak risk) —
# kept only as metadata for the spatial-block split below.
FEATURES = [
    "bright_ti4", "bright_ti5", "frp", "scan", "track",
    "brightness_temp_diff", "frp_to_temp_ratio", "footprint_proxy",
    "temp_percentile_in_region",
    "persistence_count_7d", "persistence_count_30d",
    "persistence_count_90d", "persistence_count_365d",
    "location_night_fraction",
    "dist_to_industrial_km", "dist_to_farmland_km", "dist_to_mining_km",
    "month", "day_of_year", "hour",
    "confidence_enc", "daynight_enc", "satellite_enc", "instrument_enc",
    "land_cover_class_enc", "industrial_polygon_type_enc",
    "blue", "green", "red", "nir", "swir1", "swir2",
    "ndvi", "ndbi", "nbr",
    "satellite_data_available", "is_vegetated", "is_built_up", "burn_signal",
]
FEATURES = [f for f in FEATURES if f in df.columns]
print(f"Using {len(FEATURES)} features")

X = df[FEATURES].copy()
y = df["category"].copy()

label_encoder = LabelEncoder()
y_enc = label_encoder.fit_transform(y)
print("Classes:", list(label_encoder.classes_))

Using 39 features
Classes: ['agricultural', 'flare', 'industrial', 'mining', 'offshore_flare_or_platform', 'wildfire']


## Step 4: Leakage-safe spatial-block train/test split

Splitting by `grid_cell_id` (not randomly) means detections from the SAME location
never appear in both train and test — this prevents the model from memorizing a
specific facility's coordinates instead of learning the general pattern.

**Test size: 20%** (`test_size=0.2` → 80% train / 20% test)

In [5]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y_enc, groups=df["grid_cell_id"]))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y_enc[train_idx], y_enc[test_idx]

print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")
no_overlap = len(set(df.iloc[train_idx]["grid_cell_id"]) & set(df.iloc[test_idx]["grid_cell_id"])) == 0
print("No grid-cell overlap between train/test:", no_overlap)

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

Train: 75173 rows | Test: 17696 rows


No grid-cell overlap between train/test: True


### Save outputs for the next notebook (model training)

In [6]:
# --- Notebook-chaining glue code (added when splitting the original pipeline) ---
joblib.dump(X_train, "fe_X_train.joblib")
joblib.dump(X_test, "fe_X_test.joblib")
joblib.dump(y_train, "fe_y_train.joblib")
joblib.dump(y_test, "fe_y_test.joblib")
joblib.dump(sample_weights, "fe_sample_weights.joblib")
joblib.dump(FEATURES, "fe_features.joblib")
joblib.dump(label_encoder, "fe_label_encoder.joblib")
joblib.dump(encoders, "fe_categorical_encoders.joblib")
df.to_csv("fe_output_df.csv", index=False)
print("Saved train/test splits, encoders, feature list, and the full dataframe for the model training notebook.")

Saved train/test splits, encoders, feature list, and the full dataframe for the model training notebook.
